# Notebook 03 of 7 — Basket X-Ray + Risk

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

In NB02 I built a per-name opinion of MSFT. Now I need to answer the question NB02 kept dodging: what do all 10 of my positions look like *together*? Concentration, risk, drawdown — the numbers I've been ignoring because I only have 10 tickers so 'obviously' I'm diversified.

By the end of this notebook we will be able to answer one question:

> *What am I actually exposed to at the basket level — after looking through my ETFs?*


In [ ]:
# [CODE PLACEHOLDER — Phase B] environment sanity — assert .venv_portfolio is active; STATE dir created; friendly halt with setup command if not


## 1. Load the basket

The 10-position basket from NB01. If `.notebook_state/basket.json`
doesn't exist yet (running NB03 standalone), we regenerate it from the
same locked list.

*The code cell below loads the basket and prints a positions table.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] load basket.json OR regenerate from the STORY_BIBLE locked list; print positions table (ticker, weight)


## 2. The naive view — sector pie WITHOUT look-through

Before we look through the ETFs, what does the raw sector view say?
This is what a spreadsheet would tell me: MSFT is Tech, NVDA is Tech,
AMD is Tech, QQQ is "ETF", VTI is "ETF", VNQ is "REIT", BND is "Bond
Fund", GLD is "Commodity". A rough count says maybe 38% Tech.

Hold that number.

*The code cell below classifies each position by its own sector and
renders a pie.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] compute naive sector weights (each ETF is its own row); render pie; print the raw-Tech percentage


## 3. The x-ray — look-through via `obb.portfolio_intel.xray.look_through`

Now the real view. `look_through` reads each ETF's holdings via
`EtfHoldings`, re-weights each underlying position by (basket weight ×
ETF weight-in-basket), and returns a *flattened* portfolio: what the
basket actually owns, not what it looks like on the tab labels.

QQQ's top holdings are MSFT, NVDA, AAPL, GOOGL, AMD — the exact names
already in my basket at full weight. VTI has a ~30% tech tilt in 2026
and MSFT is one of its top three positions. The moment the flattened
view lands, my "10 things" is closer to 47 things — but weighted so
that a handful of names dominate.

*The code cell below runs the look-through and prints the top-15
effective positions after flattening.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] call obb.portfolio_intel.xray.look_through on basket; render top-15 effective holdings sorted by effective weight


## 4. Sector pie WITH look-through — the pivot

Same chart as §2, but on the flattened portfolio. This is the picture
that changed how I think about my book.

Depending on when you run this against live ETF holdings, the exact
number moves, but the shape is stable: raw Tech ~38% → effective Tech
in the mid-50s. A "10-ticker diversified" basket is one sector with a
hat on.

*The code cell below re-renders the sector pie post-look-through, side
by side with the raw pie from §2.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] compute post-look-through sector weights; render side-by-side pies (raw vs xray); print delta table by sector


## 5. HHI + effective-N — two numbers Sam will quote from now on

Two concentration metrics worth learning:

- **HHI (Herfindahl-Hirschman Index)** — sum of squared weights.
  Ranges from 1/N (perfectly diversified across N names) to 1 (all in
  one name). Ours will roughly double post-look-through.
- **Effective N** — `1 / HHI`. Answers "how many *equivalent* equal-weight
  positions am I holding?" We ostensibly hold 10, but after
  look-through the effective-N number will be much lower.

These two numbers replace "I hold 10 things" as the sentence Sam
answers with when someone asks "how concentrated is your book?"

*The code cell below computes HHI and effective-N for both the raw and
the flattened views.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] compute HHI + effective-N for raw and xray views; render 2x2 table; print the interpretation ('I hold 10 things, effective N is X')


## 6. Risk metrics — `obb.portfolio_intel.risk.metrics`

Four numbers you should never operate without:

- **Sharpe ratio** — annualized excess return per unit of vol. Above 1
  is decent, above 2 is suspicious, above 3 is either a genius or a
  bug.
- **Annualized volatility** — the ± you should expect on your book.
- **Maximum drawdown** — the worst peak-to-trough decline over the
  lookback. Ask yourself: could I actually sit through this?
- **Tracking error vs SPY** — how far your book wanders from the
  benchmark. High tracking error is only worth it if your Sharpe
  clears the benchmark's.

*The code cell below computes all four on the basket and renders them
against SPY for context.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] call obb.portfolio_intel.risk.metrics on basket; compute same for SPY; render 2-column table (basket, SPY) with the 4 metrics


## 7. Concentration — `obb.portfolio_intel.risk.concentration`

Rounds out the picture with:

- **Top-K weights** (K = 1, 3, 5) — how much of the book is in the
  top handful of effective positions
- **Single-name kill-shot** — what happens to portfolio value if the
  single largest effective position drops 20%

For a basket that *looks* like 10 positions but is effectively
5-6 mega-cap tech names, the top-1 weight is going to surprise you.

*The code cell below runs `.concentration` and prints the top-K table
plus the kill-shot number.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] call obb.portfolio_intel.risk.concentration; render top-K table + kill-shot


## 8. Silent-failure guards — why the numbers can be trusted

`obb.portfolio_intel.risk` was hardened in PR #905 against three ways
these metrics silently return garbage:

1. **Partial-book variance** — if only 7 of 10 positions have valid
   price history, the covariance matrix is undefined. Old code
   substituted a rank-deficient matrix; new code refuses and tells
   you which positions failed.
2. **NaN in covariance / returns / benchmark** — same-shape refusal.
3. **NaN / Inf prices in the input** — refused at the door.

*The code cell below deliberately corrupts one price series in the
basket and re-runs `.metrics` to show the refusal path fires (not a
silent-zero result).*

In [ ]:
# [CODE PLACEHOLDER — Phase B] clone basket, poison one price series with NaN; call .metrics; assert exception raised OR guard-flag set; print the exact refusal


## 9. Reading the numbers

A quick reference — the two-sentence version of each metric, in Sam's
words:

- **HHI**: sum of squared weights. Higher means more concentrated. Ours
  roughly doubles when we look through the ETFs.
- **Effective N**: 1/HHI. "How many equal-weight positions am I really
  holding?"
- **Sharpe**: excess return over risk-free per unit of vol. Under
  ~0.5, your risk isn't being paid for.
- **Max drawdown**: worst historical peak-to-trough. If it hurts to
  read the number, it will hurt more to live through it.
- **Tracking error**: how far you wander from your benchmark. Only worth
  it if your Sharpe beats the benchmark's.

## 10. Save state for NB04, NB05, NB06

Everything downstream picks up:
- `.notebook_state/basket.json` — the 10 positions (already written)
- `.notebook_state/xray.pkl` — the flattened positions + sector weights
- `.notebook_state/risk.pkl` — the four risk metrics

*The code cell below pickles the x-ray + risk artifacts.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] pickle xray + risk objects to .notebook_state/; print sizes + keys


---

## What is NOT in this notebook

- **Cash positions.** Sam holds equity + ETFs only in this basket; cash-as-a-position support is [#903](https://github.com/prajoria/OpenBB/issues/903) and not shipped.
- **Short positions.** Same story — [#904](https://github.com/prajoria/OpenBB/issues/904).
- **Factor decomposition (Fama-French, Carhart).** The `openbb_famafrench` extension exists but isn't wired into portfolio_intel risk yet.

## Preview of NB04

Now I know what I own and how concentrated I am. The picture is worse than I thought. But the picture is static — it's a snapshot of my exposures. Two things move that snapshot every week: **events** on the calendar (earnings, dividends, splits) and **smart-money activity** (13F changes, insider transactions, government trades). In NB04 we overlay both.
